## SETUP

In [ ]:
import time

import sys

import logging

import os

import importlib

from subprocess import Popen

from IPython.display import display, clear_output

from tqdm.notebook import trange

sys.path.append('./src')

import bot_core

import bot_perception

import port_scan

import bot_handler

[importlib.reload(lib) for lib in [
    bot_core,
    bot_perception,
    port_scan,
    bot_handler
]]

logger = logging.getLogger(__name__)

# Find device (should be emulator-5554 but some installations are weird)
device = port_scan.get_device()
if device is None:
    raise Exception("No device found!")

# Start Scrcpy once per restart
if 'started_scrcpy' not in vars():
    started_scrcpy = True
    proc = Popen(['.scrcpy/scrcpy', '-s', device], shell=True)
    time.sleep(1)  # <-- sleep for 1 second
    proc.terminate()  # <-- terminate the process (Scrcpy window closed)

# Selected units in deck (order does not matter)
sel_units = [
    'chemist.png',
    'knight_statue.png',
    'harlequin.png',
    'dryad.png',
    'demon_hunter.png'
]

bot_handler.select_units(sel_units)
bot = bot_core.Bot(device)


In [ ]:
if os.path.exists('cv-images/all_units'):
	print(os.listdir('cv-images/all_units'))
else:
	print("Directory 'cv-images/all_units' does not exist")


## RUN BOT

## TRAIN RANK MODEL

In [ ]:
import sys
from pathlib import Path

# Ensure imports work in notebook
sys.path.insert(0, str(Path('./Src').resolve()))
import bot_perception

# (Optional) append current OCR_inputs samples to dataset (auto-labeled)
bot_perception.add_grid_to_dataset()

# Train & save model
model = bot_perception.train_rank_model()
out = bot_perception.save_rank_model(model)
print('Saved:', out)
print('Classes:', list(model.classes_))

In [ ]:
# Run the bot using the handler
bot_handler.run_bot(bot, max_rounds=100)



## Quick restart

In [ ]:
# Drop co-op game
bot.restart_game(quick_disconnect=True)

In [ ]:
# Restart game
bot.restart_game()